In [ ]:
from sklearn.model_selection import GridSearchCV
from sklearn.naive_bayes import BernoulliNB
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
import numpy as np
import pandas as pd

# from sklearn.model_selection import train_test_split
# from sklearn.ensemble import RandomForestClassifier
# from sklearn.metrics import f1_score
# from sklearn.decomposition import PCA

# Step 1: Data Preprocessing
data = pd.read_csv('train.csv')
data1 = pd.read_csv('test.csv')
data.drop(columns=['ID','Candidate','Constituency ?'], inplace=True)
data1.drop(columns=['ID','Candidate','Constituency ?'], inplace=True)

def convert_to_lakhs(value):
    value = str(value)
    if value == "0":
        return 0
    elif 'Crore+' in value:
        number, unit = value.split()
        number = float(number.replace(",", ""))
        return int(number * 100)
    elif 'Lac+' in value:
        number, unit = value.split()
        number = float(number.replace(",", ""))
        return int(number)
    elif 'Thou+' in value:
        number, unit = value.split()
        number = float(number.replace(",", ""))
        return int(number / 100)
    elif 'Hund+' in value:
        number, unit = value.split()
        number = float(number.replace(",", ""))
        return int(number / 1000)
    else:
        return int(value)

data["Total Assets"] = data["Total Assets"].apply(convert_to_lakhs)
data["Liabilities"] = data["Liabilities"].apply(convert_to_lakhs)
data1["Total Assets"] = data1["Total Assets"].apply(convert_to_lakhs)
data1["Liabilities"] = data1["Liabilities"].apply(convert_to_lakhs)

# Perform label encoding for Education column

from sklearn.preprocessing import LabelEncoder

# Encode the target variable
label_encoder = LabelEncoder()
data['Education'] = label_encoder.fit_transform(data['Education'])

# data['Education'] = data['Education'].astype('category').cat.codes

data = pd.get_dummies(data, columns=['Party','state'])
data1 = pd.get_dummies(data1, columns=['Party','state'])

label_counts = data['Education'].value_counts()  # Count the occurrences of each label
total_samples = len(data)  # Total number of samples in the column
label_probabilities = [label_counts.get(i, 0) / total_samples for i in range(10)]

print(label_probabilities)

X = data.drop(columns=['Education'])
y = data['Education']

# Vectorize the text data
# vectorizer = CountVectorizer(binary=True)
# X_train = vectorizer.fit_transform(X_train)
# data1 = vectorizer.transform(data1)
# Split data into train and test sets
X_train_split, X_test_split, y_train_split, y_test_split = train_test_split(X, y, test_size=0.2, random_state=42)

In [ ]:
data1

In [ ]:
# alpha_range = np.arange(0, 1.01, 0.05)  # Values of alpha from 0 to 1 with a precision of 0.01
# binarize_range = [None] + list(np.arange(0, 1.01, 0.05))  # Values of binarize from 0 to 1 with a precision of 0.01, including None
# fit_prior = [False, True]  # Keep fit_prior as True
# class_prior = [None] + list(label_probabilities)
# # Define parameter grid
# param_grid = {
#     'alpha': alpha_range,
#     'binarize': binarize_range,
#     'fit_prior': fit_prior,
#     'class_prior': label_probabilities
# }

# # Initialize Bernoulli Naive Bayes model
# model = BernoulliNB()

# # GridSearchCV
# grid_search = GridSearchCV(estimator=model, param_grid=param_grid, cv=5, scoring='f1_weighted', n_jobs=-1)
# grid_search.fit(X_train, y_train)

# # Best hyperparameters
# best_params = grid_search.best_params_
# print("Best hyperparameters:", best_params)

In [ ]:
model = BernoulliNB(alpha=0.5, binarize=0.1, fit_prior=True, class_prior=None)

model.fit(X_train_split, y_train_split)
y_pred = model.predict(data1)
y_pred = label_encoder.inverse_transform(y_pred)
y_pred

In [ ]:
# # Predict on test data using best model
# best_model = grid_search.best_estimator_
# y_pred = best_model.predict(data1)
# y_pred = label_encoder.inverse_transform(y_pred)

In [ ]:
df_pred = pd.DataFrame(y_pred, columns=["Education"])
df_pred.to_csv('Bernoullli.csv', index_label='ID')